In [1]:
from upath import UPath
from parkrun_scraper_sdk import ParkrunDataExtractionOrchestrator, Country, Course, CountriesHandler, CoursesHandler, ProcessingConfig, ResultsHandler, EventsHandler#, RunnersHandler



In [2]:
processing_date = "2015-12-24"
country_ids=["3"]
course_ids=[]

# country_ids=[#'3','4','14',
#     '23'
#     #,'30','31','32','42','44','46','54','57','64','65','67','74','82','85','88','97','98'
#              ]
# course_ids=[]

base_path = UPath("/home/nathanielramm/parkrun_data")

config = ProcessingConfig(
    base_path=base_path,
    processing_date=processing_date,
    country_ids=country_ids,
    course_ids=course_ids
)
config.normalize_ids()
config.validate()

event_orchestrator = ParkrunDataExtractionOrchestrator(config=config)

event_orchestrator.update_countries()
event_orchestrator.update_courses()
#TODO: Notify of changes!


courses_to_process = event_orchestrator.get_courses_in_config_scope()



#TODO: Need tp store last_updated date for each course. Only process if there has been a Saturday event since the last update in the given country!

#Update events
for course in courses_to_process:
    event_orchestrator.events_handler.update_event_history(course=course)


KeyboardInterrupt: 

In [3]:
#Update results
for course in courses_to_process:
    event_orchestrator.results_handler.process_event_results(events_handler=event_orchestrator.events_handler, course=course)

Error getting processed IDs: expected at least 1 source

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'select'
	[3] 'unique'

processed_result_event_ids: []
unprocessed_result_events: ['110', '70', '113', '125', '7', '92', '56', '35', '11', '95', '126', '25', '37', '28', '83', '82', '6', '9', '64', '58', '1', '109', '53', '55', '114', '98', '31', '18', '124', '89', '81', '29', '120', '132', '79', '42', '23', '107', '17', '69', '119', '10', '49', '47', '99', '4', '66', '40', '43', '76', '12', '68', '59', '32', '5', '104', '61', '57', '121', '45', '34', '91', '41', '63', '100', '62', '108', '87', '123', '15', '44', '97', '122', '94', '19', '71', '80', '30', '105', '21', '103', '13', '51', '131', '88', '128', '39', '72', '117', '85', '115', '8', '46', '20', '127', '106', '24', '14', '84', '26', '74', '60', '48', '129', '54', '112', '67', '101', '102', '50', '111', '86', '73', '27', '36', '3', '16', '93', '130', '90', '75', '65', '52', '33', '118', '77', '

In [ ]:
orchestrator.countries_handler.get_raw_countries_ids()
# orchestrator.courses_handler.get_raw_courses_by_country_id(country_id="14")


In [ ]:

courses_to_process

In [ ]:
#Update events
for course in courses_to_process:
    orchestrator.events_handler.update_event_history(course=course)

In [ ]:
text = "VM100+"
text[2:]


In [ ]:
import duckdb

# duckdb.sql("SELECT * FROM '/home/nathanielramm/parkrun_data/results/country_id=4/course_id=*/*.parquet'", params={"union_by_name": True})
all_results = duckdb.read_parquet(file_glob="/home/nathanielramm/parkrun_data/results/country_id=*/course_id=*/*.parquet", union_by_name=True)

all_results.s


In [11]:
import ibis
from ibis import _

dbconn = ibis.duckdb.connect(database="/home/nathanielramm/parkrun_data/parkrun.duckdb")



In [ ]:
tbl_results_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/results/country_id=*/course_id=*/*.parquet', union_by_name=True)
tbl_events_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/events/country_id=*/*.parquet', union_by_name=True)
tbl_courses_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/courses/courses.parquet', union_by_name=True)
tbl_countries_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/countries/countries.parquet', union_by_name=True)



In [ ]:
# runners = ( tbl_results_raw
#            .group_by(["athlete_id", "gender"])
#             .aggregate(total_runs=_.gender.count())
#             .select(["athlete_id", "gender"])
#            .group_by(["athlete_id"])
#             .aggregate(total_genders=_.gender.count())
#             .order_by(ibis.desc("total_genders"))
# )

# runners.execute()

tbl_results_raw.group_by(["age_group"]).aggregate(total_runs=_.athlete_id.count()).order_by(ibis.desc("total_runs")).execute()

In [ ]:

def format_time(time_col):

    clean_time = time_col.replace(':', '')
    # Convert to string and pad left with zeros to ensure 6 digits
    padded = clean_time.cast('string').lpad(6, '0')
    
    # Extract hours, minutes, seconds
    hours = padded.substr(0, 2)
    minutes = padded.substr(2, 2)
    seconds = padded.substr(4, 2)
    
    # Combine with colons
    return hours + ':' + minutes + ':' + seconds


def time_to_seconds(time_col):
    # Split on colons and get components

    parts = time_col.split(':')
    hours = parts[0].cast('int64')
    minutes = parts[1].cast('int64')
    seconds = parts[2].cast('int64')
    
    # Convert to total seconds
    # hours * 3600 + minutes * 60 + seconds
    return (hours * 3600) + (minutes * 60) + seconds


v_lkp_course_series = tbl_courses_raw.select(["course_id", "series_id"]).distinct()


v_events_raw = (tbl_events_raw.left_join(v_lkp_course_series, predicates=[tbl_events_raw.course_id == v_lkp_course_series.course_id])
                .select(["course_id","country_id","event_id","event_date", "series_id",
                         "finishers","volunteers" ])
                .mutate(course_id=tbl_events_raw.course_id.cast(target_type=str),
                        country_id=tbl_events_raw.country_id.cast(target_type=str),
                        event_id=tbl_events_raw.event_id.cast(target_type=str),
                        event_id_int=tbl_events_raw.event_id.cast(target_type=int),                        
                        event_date=ibis.date(tbl_events_raw.event_date),
                        series_id=v_lkp_course_series.series_id,

                        finishers=tbl_events_raw.finishers.cast(target_type=int),
                        volunteers=tbl_events_raw.volunteers.cast(target_type=int)
                        )
                )

v_event_winners_raw = (tbl_events_raw.left_join(right=v_lkp_course_series, predicates=[tbl_events_raw.course_id == v_lkp_course_series.course_id])
                       
                .select(["course_id","country_id","event_id","event_date", "series_id", 
                         "male_first_athlete_name","female_first_athlete_name", "male_time", "female_time", "male_athlete_number", "female_athlete_number"])
                .mutate(course_id=tbl_events_raw.course_id.cast(target_type=str),
                        country_id=tbl_events_raw.country_id.cast(target_type=str),
                        event_id=tbl_events_raw.event_id.cast(target_type=str),
                        event_id_int=tbl_events_raw.event_id.cast(target_type=int),                        
                        event_date=ibis.date(tbl_events_raw.event_date),

                        male_athlete_number=tbl_events_raw.male_athlete_number.cast(target_type=str),
                        female_athlete_number=tbl_events_raw.female_athlete_number.cast(target_type=str),

                        male_first_athlete_name=tbl_events_raw.male_first_athlete_name.cast(target_type=str),
                        female_first_athlete_name=tbl_events_raw.female_first_athlete_name.cast(target_type=str),

                        male_time_raw=tbl_events_raw.male_time.cast(target_type=str),
                        female_time_raw=tbl_events_raw.female_time.cast(target_type=str),

                        male_time_corrected = format_time(time_col=tbl_events_raw.male_time),
                        female_time_corrected = format_time(time_col=tbl_events_raw.female_time)
                )
                .mutate(

                        male_time_seconds = time_to_seconds(time_col=_.male_time_corrected),
                        female_time_seconds = time_to_seconds(time_col=_.female_time_corrected)
                )
)

# v_event_winners_raw.execute()




v_event_winners_raw.filter(v_event_winners_raw.series_id == "1", v_event_winners_raw.male_time_seconds.notnull()).order_by(ibis.asc("male_time_seconds")).execute()

# v_event_winners_raw.filter(v_event_winners_raw.series_id == "1").aggregate(by=["country_id", "course_id", "series_id"],  record_male_winner_time=_.male_time_seconds.min()).order_by(ibis.asc("record_male_winner_time")).execute()


                        # male_time_seconds=ibis.cast(ibis.split_part(tbl_events_raw.male_time, ":", 1) * 60 + ibis.split_part(tbl_events_raw.male_time, ":", 2), target_type=int),
                        # female_time_seconds=ibis.cast(ibis.split_part(tbl_events_raw.female_time, ":", 1) * 60 + ibis.split_part(tbl_events_raw.female_time, ":", 2), target_type=int),


In [ ]:
tbl_courses_raw.filter(tbl_courses_raw.course_id == "452").execute()

In [ ]:

v_country_id_max_date = v_events_raw.group_by("country_id").aggregate(max_country_date=_.event_date.max()).order_by(ibis.asc("max_country_date"))
v_course_id_max_date = v_events_raw.group_by(["course_id", "country_id"]).aggregate(max_event_date=_.event_date.max()).order_by(ibis.asc("max_event_date"))
v_course_max_event_date = v_course_id_max_date.left_join(v_country_id_max_date, predicates=[v_course_id_max_date.country_id == v_course_id_max_date.country_id]).mutate(date_diff=v_country_id_max_date.max_country_date - v_course_id_max_date.max_event_date)

v_course_max_event_date.filter(v_course_max_event_date.date_diff > 6).execute()
# course_id_max_date.execute()

In [ ]:
tbl_results_raw.group_by("athlete_id").aggregate("athlete_id").count()

In [ ]:

# orchestrator.events_handler.get_processed_event_ids(course=course)
# orchestrator.events_handler.get_raw_course_event_ids(course=course)

# orchestrator.results_handler.process_event_results(events_handler=orchestrator.events_handler, course=course, event=event)


# orchestrator.results_handler.get_stored_result_event_ids(course=course)
